# Experiment 03: Series mean vs building embedding

Card: [exp03_mean_vs_embedding.md](../../experiments/exp03_mean_vs_embedding.md)

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
from ashrae.config import FOLDS, METERS
from ashrae.data import read_processed, split
from ashrae.features import add_series_mean, build_features
from ashrae.metrics import rmsle, rmsle_by_meter
from ashrae.modeling import MeterMLP, make_loaders, predict
from ashrae.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
NOTEBOOK = "exp03_mean_vs_embedding"
MODELS = ["mean", "emb"]
if SMOKE:
    EXPERIMENT = "ashrae-validation-mean_vs_embedding-smoke"
    TRAIN = True
    FOLD_NAMES = list(FOLDS)[:2]
    N_TRAIN, N_VAL = 20_000, 5_000
    EPOCHS = 1
    FIGURE = None
else:
    EXPERIMENT = "ashrae-validation-mean_vs_embedding"
    TRAIN = False  # the runs below are from 2026-09-27; True retrains them under a new session
    FOLD_NAMES = list(FOLDS)
    N_TRAIN, N_VAL = 2_000_000, 500_000
    EPOCHS = 10
    FIGURE = "../../figures/exp03_mean_vs_embedding.png"
SESSION = None if TRAIN else "2026-09-27 13:26:10"

experiment = Experiment(EXPERIMENT, NOTEBOOK, SESSION)

## Training

In [ ]:
def fold_data(val_months):
    train_df, val_df = split(df, val_months, N_TRAIN, N_VAL)
    year_median = train_df.year_built.median()
    X_train = add_series_mean(build_features(train_df, year_median), train_df, train_df)
    X_val = add_series_mean(build_features(val_df, year_median), val_df, train_df)
    no_mean = lambda X: X.drop(columns="series_mean")
    return {
        "mean": make_loaders(train_df, val_df, X_train, X_val)[:2],
        "emb": make_loaders(train_df, val_df, no_mean(X_train), no_mean(X_val))[:2],
        "anomaly": val_df.anomaly.to_numpy(),
        "meter": val_df.meter.to_numpy(),
    }


if TRAIN:
    df = read_processed()
    n_buildings = int(df.building_id.max()) + 1
    folds = {fold: fold_data(FOLDS[fold]) for fold in FOLD_NAMES}
    del df
    n_features = folds[FOLD_NAMES[0]]["emb"][0].dataset.x.shape[1]
    # the two differ only in how they know the building: one extra input, or a 16-number vector per building
    make_net = {"mean": lambda: MeterMLP(n_features + 1, batch_norm=True), "emb": lambda: MeterMLP(n_features, n_buildings, batch_norm=True)}
    for fold, data in folds.items():
        anomaly, meter = data["anomaly"], data["meter"]
        for model_name in MODELS:
            train_loader, val_loader = data[model_name]
            model, run_id = experiment.fit(
                make_net[model_name], train_loader, run_name=f"{model_name} | {fold}", epochs=EPOCHS,
                params={"n_train": N_TRAIN, "n_val": N_VAL}, tags={"fold": fold, "model": model_name},
            )
            pred, y = predict(model, val_loader), val_loader.dataset.y.numpy()
            scores = {"train": experiment.history(run_id, "train_rmsle")[-1],
                      "val_all": rmsle(y, pred), "val_clean": rmsle(y[~anomaly], pred[~anomaly])}
            meters = rmsle_by_meter(y[~anomaly], pred[~anomaly], meter[~anomaly], prefix="clean")
            experiment.log_metrics(run_id, {**{f"rmsle_{k}": v for k, v in scores.items()}, **meters})

## Results

In [ ]:
runs = experiment.runs().rename(columns=lambda c: c.removeprefix("tags.").removeprefix("metrics."))
print(f"MLflow session {experiment.session}: {len(runs)} runs")
results = runs[["fold", "model", "rmsle_train", "rmsle_val_all", "rmsle_val_clean"]].rename(
    columns=lambda c: c.removeprefix("rmsle_"))
by_meter = runs.melt(id_vars=["fold", "model"], value_vars=[f"clean_{m}" for m in METERS.values()],
                     var_name="meter", value_name="val_clean")
by_meter["meter"] = by_meter.meter.str.removeprefix("clean_")

# difference = mean − emb: positive means the mean model is worse
table = results.pivot(index="fold", columns="model", values=["val_clean", "val_all"]).reindex(FOLD_NAMES)
table.loc["average"] = table.mean()
for score in ["val_clean", "val_all"]:
    table[(score, "mean − emb")] = table[(score, "mean")] - table[(score, "emb")]
display(table[[(s, c) for s in ["val_clean", "val_all"] for c in [*MODELS, "mean − emb"]]].round(3))

meter_table = by_meter.pivot_table(index="meter", columns="model", values="val_clean").reindex(list(METERS.values()))
meter_table["mean − emb"] = meter_table["mean"] - meter_table["emb"]
display(meter_table.round(3).rename_axis("clean RMSLE, average over folds"))

In [ ]:
ax = table.loc[FOLD_NAMES, "val_clean"][MODELS].plot.bar(figsize=(7, 4), rot=0, width=0.8)
ax.set(title="Clean validation RMSLE by held-out period", xlabel="held-out months", ylabel="RMSLE")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=7)
ax.legend(["series mean", "embedding"], fontsize=8)
plt.tight_layout()
if FIGURE:
    ax.figure.savefig(FIGURE, dpi=150, bbox_inches="tight")
    for run_id in runs.run_id:
        experiment.client.log_artifact(run_id, FIGURE)